In [4]:
import os
import uuid
from typing import List
from dotenv import load_dotenv

# Foundation imports (the ones that ARE working)
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(".env")

# 1. OUR CUSTOM RETRIEVER (Using a simple Python Dict instead of InMemoryStore)
class SimpleParentRetriever(BaseRetriever):
    vectorstore: Chroma
    # We replace InMemoryStore with a standard Python dictionary
    docstore: dict 
    id_key: str = "doc_id"

    def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        # 1. Search for the small "Child" chunks
        child_chunks = self.vectorstore.similarity_search(query, k=4)
        
        # 2. Get unique Parent IDs from the metadata
        parent_ids = {doc.metadata[self.id_key] for doc in child_chunks if self.id_key in doc.metadata}
        
        # 3. Pull the full Parent text from our dictionary
        return [self.docstore[pid] for pid in parent_ids if pid in self.docstore]

# 2. INITIALIZE MODELS
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. SETUP STORAGE
vectorstore = Chroma(collection_name="safe_storage", embedding_function=embeddings)
parent_dictionary = {} # Standard Python Dict - cannot fail to import!

# 4. DATA INGESTION
def ingest_data(file_path):
    if not os.path.exists(file_path):
        return
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    p_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    c_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

    parents = p_splitter.create_documents([text])
    
    for p in parents:
        p_id = str(uuid.uuid4())
        # Link children to this parent
        children = c_splitter.split_documents([p])
        for c in children:
            c.metadata["doc_id"] = p_id
        
        vectorstore.add_documents(children)
        parent_dictionary[p_id] = p # Save to our dictionary

ingest_data("sample.txt")

# 5. INITIALIZE THE RETRIEVER
retriever = SimpleParentRetriever(vectorstore=vectorstore, docstore=parent_dictionary)

# 6. FINAL TOOL AND EXECUTION
@tool
def ask_docs(question: str) -> str:
    """Queries the local documentation using a robust parent-child link."""
    docs = retriever.invoke(question)
    if not docs:
        return "No relevant information found."
    return "\n\n---\n\n".join([d.page_content for d in docs])

# TEST
print("\n🚀 System Running...")
context = ask_docs.invoke("Who created LangChain?")
response = llm.invoke(f"Context: {context}\n\nQuestion: Who created LangChain?")
print(f"\n🤖 ANSWER:\n{response.content}")


🚀 System Running...

🤖 ANSWER:
Harrison Chase


In [1]:
#BM25 RETRIEVER
import os
from dotenv import load_dotenv

# Stable imports
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

# The BM25 Retriever lives in Community
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(".env")

# 1. INITIALIZE LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# 2. LOAD AND SPLIT DATA
def prepare_bm25():
    if not os.path.exists("sample.txt"):
        print("❌ sample.txt not found!")
        return None
        
    with open("sample.txt", "r", encoding="utf-8") as f:
        text = f.read()
    
    # BM25 works best with slightly larger chunks to capture keywords
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    docs = splitter.create_documents([text])
    
    # Initialize BM25 directly from documents
    # This does NOT require a vector database or embeddings!
    retriever = BM25Retriever.from_documents(docs)
    
    # Set k (how many chunks to return)
    retriever.k = 3
    return retriever

bm25_retriever = prepare_bm25()

# 3. AGENT TOOL
@tool
def ask_docs_bm25(question: str) -> str:
    """Uses keyword matching (BM25) to find exact terms in documents."""
    if not bm25_retriever:
        return "Knowledge base not initialized."
        
    docs = bm25_retriever.invoke(question)
    return "\n\n---\n\n".join([d.page_content for d in docs])

# 4. EXECUTION
print("\n🚀 BM25 System Ready.")
query = "Who created LangChain?"

# Get context
context = ask_docs_bm25.invoke(query)

# Ask LLM
response = llm.invoke(f"Context: {context}\n\nQuestion: {query}")
print(f"\n🤖 ANSWER:\n{response.content}")


🚀 BM25 System Ready.

🤖 ANSWER:
Harrison Chase


In [2]:
import os
from dotenv import load_dotenv

# Foundation imports
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Try to import Ensemble directly, with a fallback just in case
try:
    from langchain.retrievers import EnsembleRetriever
except (ImportError, ModuleNotFoundError):
    # If the import fails, we use a simple list-based merge
    EnsembleRetriever = None 

load_dotenv(".env")

# 1. INITIALIZE MODELS
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. PREPARE DATA
with open("sample.txt", "r", encoding="utf-8") as f:
    text = f.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
docs = splitter.create_documents([text])

# 3. INITIALIZE THE TWO RETRIEVERS
# A. BM25 (Keyword Search - No embeddings needed)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

# B. Vector (Semantic Search - Needs embeddings)
vectorstore = Chroma.from_documents(docs, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 4. SETUP THE ENSEMBLE
if EnsembleRetriever:
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever], 
        weights=[0.5, 0.5]
    )
else:
    # Fallback: A simple manual merger if the library is broken
    def ensemble_retriever_fallback(query):
        bm25_docs = bm25_retriever.invoke(query)
        vec_docs = vector_retriever.invoke(query)
        return list({d.page_content: d for d in (bm25_docs + vec_docs)}.values())

# 5. AGENT TOOL
@tool
def ask_hybrid_docs(question: str) -> str:
    """Uses Hybrid Search (Vector + BM25) to find exact and conceptual matches."""
    if EnsembleRetriever:
        results = ensemble_retriever.invoke(question)
    else:
        results = ensemble_retriever_fallback(question)
        
    return "\n\n---\n\n".join([res.page_content for res in results])

# 6. EXECUTION
print("\n🚀 Hybrid System (Vector + BM25) is online.")
query = "Who created LangChain?"
context = ask_hybrid_docs.invoke(query)
response = llm.invoke(f"Context: {context}\n\nQuestion: {query}")

print(f"\n🤖 ANSWER:\n{response.content}")


🚀 Hybrid System (Vector + BM25) is online.

🤖 ANSWER:
Harrison Chase
